In [2]:
# %% Cell: Load Libraries and Setup
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Define UGA-inspired color palette (reds, grays, neutrals, black)
uga_palette = {"red": "#BA0C2F", "gray": "#7E7E7E", "neutral": "#CCCCCC", "black": "#000000"}  # UGA red  # UGA gray  # Neutral tone  # Black

# Set Seaborn palette and matplotlib style for consistency in future visualizations
sns.set_palette(list(uga_palette.values()))
sns.set_theme(style="whitegrid")

# Devil's advocate: Ensure these styles align with the journal’s requirements.
print("Libraries loaded and UGA palette set using sns.set_theme(style='whitegrid').")

# %% Cell: Load Data
# Load the initial dataset (meyer.csv) and the aggregated (cleaned) dataset (meyer_aggregated_binary_renamed.csv)
# Make sure these CSV files are in your working directory.
try:
    df_initial = pd.read_csv("meyer.csv")
    df_aggregated = pd.read_csv("meyer_aggregated_binary_renamed.csv")
    print("Data loaded successfully.")
except Exception as e:
    print("Error loading data:", e)

# %% Cell: Data Overview and Type Conversion
# Display basic information for both datasets to understand structure and contents.
print("Initial Dataset Info:")
print(df_initial.info())
print(df_initial.head())

print("\nAggregated Dataset Info:")
print(df_aggregated.info())
print(df_aggregated.head())

# Convert factor columns to proper types.
# For our experiment, these factors should be treated as categorical.
factor_columns = ["moisture", "spring_stiffness", "displacement_screw_setting", "motor_speed"]
for col in factor_columns:
    if col in df_initial.columns:
        df_initial[col] = df_initial[col].astype("category")
    if col in df_aggregated.columns:
        df_aggregated[col] = df_aggregated[col].astype("category")

# Devil's advocate: Verify if the factors should indeed be categorical or if any can be treated as numeric.
print("Factor columns converted to categorical types where applicable.")

# %% Cell: Check for Out-of-Range or Unexpected Values
# Define expected levels for each factor based on the experimental design.
expected_levels = {"moisture": [5, 7, 9], "spring_stiffness": [1800, 2000, 2200], "displacement_screw_setting": [0.22, 0.29, 0.36], "motor_speed": [30, 45, 60]}  # moisture percentages  # in N/mm  # in inches  # in Hz

# Devil's advocate: Confirm that our dataset adheres to these levels.
for col, levels in expected_levels.items():
    if col in df_initial.columns:
        # Convert categories to numeric for accurate comparison
        try:
            unique_vals = sorted([float(val) for val in df_initial[col].cat.categories])
        except Exception as e:
            unique_vals = sorted(df_initial[col].cat.categories)
        print(f"Unique values in {col} (initial dataset):", unique_vals)
        for val in unique_vals:
            if val not in levels:
                print(f"Warning: Value {val} in column '{col}' is outside expected levels {levels}")

# %% Cell: Handle Missing Data and Missing Videos Flag
# Check for missing values in both datasets.
print("\nMissing values in the initial dataset:")
print(df_initial.isnull().sum())

print("\nMissing values in the aggregated dataset:")
print(df_aggregated.isnull().sum())

# Look for a column indicating missing videos (assumed to be 'missing_videos_flag').
if "missing_videos_flag" in df_aggregated.columns:
    # Assuming the flag is binary (1 = missing, 0 = complete)
    missing_flag_count = df_aggregated["missing_videos_flag"].sum()
    print(f"Number of runs with missing videos: {missing_flag_count}")
    # Devil's advocate: Consider whether to impute or exclude these runs.
    # Here, we create a cleaned version by excluding flagged runs.
    df_aggregated_clean = df_aggregated[df_aggregated["missing_videos_flag"] == 0].copy()
    print(f"Aggregated dataset after excluding runs with missing videos: {df_aggregated_clean.shape}")
else:
    df_aggregated_clean = df_aggregated.copy()
    print("No 'missing_videos_flag' column found; using the full aggregated dataset.")

# Check for missing values in critical columns (e.g., factor columns and run identifiers).
critical_columns = factor_columns + ["run_number"]  # assuming 'run_number' exists in aggregated data
for col in critical_columns:
    if col in df_aggregated_clean.columns:
        missing_count = df_aggregated_clean[col].isnull().sum()
        if missing_count > 0:
            print(f"Column '{col}' has {missing_count} missing values.")

# Devil's advocate: Missing data in critical columns can bias results.
# For this analysis, we choose to drop rows with missing critical data.
df_aggregated_clean.dropna(subset=critical_columns, inplace=True)

# %% Cell: Validate Experimental Design Combinations
# Check that the aggregated dataset has the expected 81 unique factor combinations.
if "run_number" in df_aggregated_clean.columns:
    unique_combinations = df_aggregated_clean[factor_columns].drop_duplicates()
    num_combinations = unique_combinations.shape[0]
    print(f"Number of unique factor combinations: {num_combinations}")
    if num_combinations != 81:
        print("Warning: The number of unique combinations is not 81. Please verify the experimental design or data coding.")
    else:
        print("Experimental design validated: 81 unique combinations found.")
else:
    print("'run_number' column not found. Skipping experimental design validation.")

# Devil's advocate: Confirm that the replicates per combination are as expected (should be 3 per combination for a total of 243 runs).
replicate_counts = df_aggregated_clean.groupby(factor_columns).size()
print("Replicate counts per combination (frequency of occurrence):")
print(replicate_counts.value_counts())

# %% Cell: Final Checks and Export Cleaned Data
# Provide a final overview of the cleaned aggregated dataset.
print("\nCleaned Aggregated Dataset Overview:")
print(df_aggregated_clean.info())
print(df_aggregated_clean.head())

# Optionally, save the cleaned dataset for use in subsequent phases.
df_aggregated_clean.to_csv("meyer_aggregated_cleaned.csv", index=False)
print("Cleaned aggregated dataset saved as 'meyer_aggregated_cleaned.csv'.")

# End of Phase A
print("\nPhase A: Data Cleaning & Preprocessing is complete. Please review the outputs and confirm before proceeding to Phase B.")

Libraries loaded and UGA palette set using sns.set_theme(style='whitegrid').
Data loaded successfully.
Initial Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 243 entries, 0 to 242
Data columns (total 15 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   moisture                        243 non-null    int64  
 1   spring_stiffness                243 non-null    int64  
 2   displacement_screw_setting      243 non-null    float64
 3   motor_speed                     243 non-null    int64  
 4   untouched                       77 non-null     object 
 5   longitudinal less than 25%      14 non-null     object 
 6   Longitudinal between 25-50%     16 non-null     object 
 7   Longitudinal between 50-75%     102 non-null    object 
 8   Longitudinal more than 75%      242 non-null    object 
 9   Circumferential less than 25%   126 non-null    object 
 10  Circumferential between 25-50%  1

KeyError: ['spring_stiffness', 'displacement_screw_setting', 'motor_speed', 'run_number']